# ADP Offline Training — Forward-Backward Algorithm

We train a **time-dependent linear Value Function Approximation (VFA)**:
$$\hat{V}_t(x_t;\, \eta_t) = \eta_t^\top \phi(x_t)$$

using Fitted Value Iteration with an initial forward pass:

1. **Forward pass** — simulate a heuristic policy over $N$ historical scenarios to collect realistic visited states $\{x_{n,t}\}$.
2. **Backward pass** — for $t = T{-}1 \ldots 0$, compute target values $V^*(x_{n,t})$ by solving a 1-step Pyomo MIQP lookahead using the already-fitted $\eta_{t+1}$.
3. **Regression** — fit $\eta_t$ by OLS: $\eta_t = \arg\min_\eta \sum_n (\eta^\top\phi(x_{n,t}) - V^*(x_{n,t}))^2$.
4. **Export** — save all $\eta_t$ ($t=0\ldots9$) to `output/adp_weights.json`.

### Basis functions $\phi(x_t) \in \mathbb{R}^7$

| Index | Feature | Motivation |
|-------|---------|------------|
| 0 | $(T_1 - T^{\mathrm{ok}})^2$ | Quadratic comfort penalty room 1 |
| 1 | $(T_2 - T^{\mathrm{ok}})^2$ | Quadratic comfort penalty room 2 |
| 2 | $H$ | Humidity level drives ventilation need |
| 3 | $c$ | Inertia counter; $c>0$ forces future ventilation cost |
| 4 | $\max(0,\, T^{\mathrm{low}} - T_1)$ | Soft cold-overrule penalty room 1 |
| 5 | $\max(0,\, T^{\mathrm{low}} - T_2)$ | Soft cold-overrule penalty room 2 |
| 6 | $1$ | Intercept |

The online policy (`ADP_policy_14.py`) reads `output/adp_weights.json` at import
time and uses $\eta_{t}$ for the current step.

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import pyomo.environ as pyo
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('.'))
import v2_SystemCharacteristics as sc

raw = sc.get_fixed_data()

params = {
    'P_max'    : raw['heating_max_power'],
    'P_vent'   : raw['ventilation_power'],
    'T_low'    : raw['temp_min_comfort_threshold'],
    'T_ok'     : raw['temp_OK_threshold'],
    'T_high'   : raw['temp_max_comfort_threshold'],
    'H_high'   : raw['humidity_threshold'],
    'zeta_exch': raw['heat_exchange_coeff'],
    'zeta_loss': raw['thermal_loss_coeff'],
    'zeta_conv': raw['heating_efficiency_coeff'],
    'zeta_cool': raw['heat_vent_coeff'],
    'zeta_occ' : raw['heat_occupancy_coeff'],
    'eta_occ'  : raw['humidity_occupancy_coeff'],
    'eta_vent' : raw['humidity_vent_coeff'],
    'T_out'    : raw['outdoor_temperature'],
}

T_INIT = float(raw['T1'])   # 21.0 °C
H_INIT = float(raw['H'])    # 40.0 %
T      = 10
N_FEAT = 7

print("System parameters loaded.")
print(f"  T_low={params['T_low']} | T_ok={params['T_ok']} | T_high={params['T_high']} | H_high={params['H_high']}")
print(f"  P_max={params['P_max']} kW | P_vent={params['P_vent']} kW")

In [ ]:
price_df = pd.read_csv('v2_PriceData.csv', header=0)
occ1_df  = pd.read_csv('OccupancyRoom1.csv', header=0)
occ2_df  = pd.read_csv('OccupancyRoom2.csv', header=0)

# v2_PriceData.csv: col0 = previous price, cols '1'..'10' = price at t=0..9
# OccupancyRoomX.csv: cols '0'..'9' = occupancy at t=0..9
price_cols = [str(i) for i in range(1, 11)]
occ_cols   = [str(i) for i in range(10)]

prices_all  = price_df[price_cols].values.astype(float)   # (N, 10)
prev_prices = price_df.iloc[:, 0].values.astype(float)    # (N,)
occ1_all    = occ1_df[occ_cols].values.astype(float)      # (N, 10)
occ2_all    = occ2_df[occ_cols].values.astype(float)      # (N, 10)

N = len(prices_all)
print(f"Loaded {N} days of historical data.")
print(f"  Price range: [{prices_all.min():.2f}, {prices_all.max():.2f}] euro/kWh")
print(f"  Occ1  range: [{occ1_all.min():.1f}, {occ1_all.max():.1f}] persons")
print(f"  Occ2  range: [{occ2_all.min():.1f}, {occ2_all.max():.1f}] persons")

In [ ]:
def phi(state):
    """
    7-dimensional feature vector.
    Matches the VFA used inside the online Pyomo model (ADP_policy_14.py).
    """
    T1  = float(state['T1'])
    T2  = float(state['T2'])
    H   = float(state.get('H', 0.0))
    c   = float(state.get('c', 0))
    T_ok  = params['T_ok']
    T_low = params['T_low']
    return np.array([
        (T1 - T_ok)**2,         # 0 – quadratic comfort deviation room 1
        (T2 - T_ok)**2,         # 1 – quadratic comfort deviation room 2
        H,                      # 2 – humidity
        c,                      # 3 – ventilation inertia counter
        max(0.0, T_low - T1),   # 4 – soft cold-overrule penalty room 1
        max(0.0, T_low - T2),   # 5 – soft cold-overrule penalty room 2
        1.0,                    # 6 – intercept
    ], dtype=float)

# Sanity check
s_test = {'T1': 21.0, 'T2': 19.0, 'H': 45.0, 'c': 0}
print("phi(test) =", phi(s_test))

In [ ]:
def apply_dynamics(state, decisions):
    """One-step transition mirroring Task6_Environment.apply_dynamics."""
    p  = params
    t  = int(state['current_time'])
    p1 = float(decisions['HeatPowerRoom1'])
    p2 = float(decisions['HeatPowerRoom2'])
    v  = int(float(decisions['VentilationON']) > 0.5)
    H  = float(state.get('H', 0.0))
    c  = int(state.get('c', 0))

    if c > 0:                  v  = 1
    if H >= p['H_high']:       v  = 1
    if state.get('y_low_1'):   p1 = p['P_max']
    if state.get('y_low_2'):   p2 = p['P_max']
    if state.get('y_high_1'):  p1 = 0.0
    if state.get('y_high_2'):  p2 = 0.0

    step_cost = float(state['price']) * (p1 + p2 + p['P_vent'] * v)

    T_out = float(p['T_out'][min(t, 9)])
    T1 = float(state['T1']); T2 = float(state['T2'])
    occ1 = float(state.get('occ1', 0.0)); occ2 = float(state.get('occ2', 0.0))

    T1_n = (T1 + p['zeta_exch']*(T2-T1) + p['zeta_loss']*(T_out-T1)
            + p['zeta_conv']*p1 - p['zeta_cool']*v + p['zeta_occ']*occ1)
    T2_n = (T2 + p['zeta_exch']*(T1-T2) + p['zeta_loss']*(T_out-T2)
            + p['zeta_conv']*p2 - p['zeta_cool']*v + p['zeta_occ']*occ2)
    H_n  = max(0.0, H + p['eta_occ']*(occ1+occ2) - p['eta_vent']*v)
    c_n  = (2 if c == 0 else max(0, c-1)) if v == 1 else 0

    y_lo1 = (1 if T1_n < p['T_low'] else
             (0 if T1_n >= p['T_ok'] else int(state.get('y_low_1', 0))))
    y_lo2 = (1 if T2_n < p['T_low'] else
             (0 if T2_n >= p['T_ok'] else int(state.get('y_low_2', 0))))
    y_hi1 = 1 if T1_n > p['T_high'] else 0
    y_hi2 = 1 if T2_n > p['T_high'] else 0

    return {
        'T1': T1_n, 'T2': T2_n, 'H': H_n, 'c': c_n,
        'y_low_1': y_lo1, 'y_low_2': y_lo2,
        'y_high_1': y_hi1, 'y_high_2': y_hi2,
        'occ1': occ1, 'occ2': occ2,
        'price': float(state['price']),
        'price_previous': float(state['price']),
        'current_time': t + 1,
    }, step_cost


def heuristic_policy(state):
    """
    Proportional rule-based heuristic for the forward pass.
    Heats proportionally to comfort deficit; ventilates when forced or humid.
    """
    T1 = float(state['T1']); T2 = float(state['T2'])
    H  = float(state.get('H', 0.0))
    c  = int(state.get('c', 0))
    P_max = params['P_max']; T_ok = params['T_ok']; T_low = params['T_low']

    v = 1 if (c > 0 or H >= params['H_high']) else int(H > 55.0)

    def heat(T_r, y_lo, y_hi):
        if y_hi: return 0.0
        if y_lo: return P_max
        if T_r >= T_ok: return 0.0
        return P_max * min(1.0, (T_ok - T_r) / (T_ok - T_low))

    return {
        'HeatPowerRoom1': heat(T1, state.get('y_low_1', 0), state.get('y_high_1', 0)),
        'HeatPowerRoom2': heat(T2, state.get('y_low_2', 0), state.get('y_high_2', 0)),
        'VentilationON' : v,
    }

print("Dynamics and heuristic policy defined.")

## Step 1 — Forward Pass

Simulate the heuristic policy over all $N$ historical scenarios.
Each scenario uses realised historical prices and occupancies,
starting from the fixed initial state $(T^{\mathrm{init}}, H^{\mathrm{init}})$.

In [ ]:
print(f"Forward pass: {N} scenarios x {T} timesteps ...")
t0 = time.perf_counter()

# states[t][n] = state dict at time t for scenario n (before decision)
states = [[None] * N for _ in range(T)]

for n in range(N):
    state = {
        'T1': T_INIT, 'T2': T_INIT, 'H': H_INIT,
        'c': 0, 'y_low_1': 0, 'y_low_2': 0, 'y_high_1': 0, 'y_high_2': 0,
        'occ1': float(occ1_all[n, 0]), 'occ2': float(occ2_all[n, 0]),
        'price': float(prices_all[n, 0]),
        'price_previous': float(prev_prices[n]),
        'current_time': 0,
    }
    for t in range(T):
        state['price']          = float(prices_all[n, t])
        state['price_previous'] = float(prices_all[n, t-1]) if t > 0 else float(prev_prices[n])
        state['occ1']           = float(occ1_all[n, t])
        state['occ2']           = float(occ2_all[n, t])
        state['current_time']   = t

        states[t][n] = dict(state)           # record state before decision
        state, _     = apply_dynamics(state, heuristic_policy(state))

print(f"Done in {time.perf_counter()-t0:.2f}s.")
T1_mid = [states[5][n]['T1'] for n in range(N)]
print(f"  T1 at t=5: mean={np.mean(T1_mid):.2f}  min={np.min(T1_mid):.2f}  max={np.max(T1_mid):.2f}")
c_mid = sorted(set(states[5][n]['c'] for n in range(N)))
print(f"  c  at t=5: {c_mid}")

## Step 2 — Backward Pass + OLS Regression

For each $t = 9, 8, \ldots, 0$ (backwards):

1. Solve the 1-step MIQP for every sampled state $x_{n,t}$:
   $$V^*(x_{n,t}) = \min_{p_1,p_2,v}\;\Bigl\{\lambda_{n,t}(p_1+p_2+P_{\text{vent}}\,v) + \eta_{t+1}^\top\phi\bigl(x_{n,t+1}(p_1,p_2,v)\bigr)\Bigr\}$$
   The VFA of the next state is expressed directly as Pyomo expressions
   through the post-decision dynamics (MIQP because of the quadratic terms).

2. Fit $\eta_t$ by OLS on $(\Phi_t, V^*_t)$.

Terminal condition: $\eta_{10} = \mathbf{0}$.

In [ ]:
def solve_1step(state, eta_next):
    """
    1-step Pyomo MIQP:
        min_{p1,p2,v}  price*(p1+p2+P_vent*v) + eta_next @ phi(x_next(p1,p2,v))

    phi(x_next) is embedded via post-decision dynamics.
    Returns the optimal objective value, or None if solver fails.
    """
    p     = params
    P_max = p['P_max']
    T_ok  = p['T_ok']
    T_low = p['T_low']
    t     = int(state.get('current_time', 0))
    T_out = float(p['T_out'][min(t, 9)])
    T1    = float(state['T1'])
    T2    = float(state['T2'])
    H     = float(state.get('H', 0.0))
    c     = int(state.get('c', 0))
    occ1  = float(state.get('occ1', 0.0))
    occ2  = float(state.get('occ2', 0.0))
    price = float(state['price'])
    eta   = eta_next

    m = pyo.ConcreteModel()
    m.p1  = pyo.Var(bounds=(0, P_max))
    m.p2  = pyo.Var(bounds=(0, P_max))
    m.v   = pyo.Var(domain=pyo.Binary)
    m.T1x = pyo.Var()
    m.T2x = pyo.Var()
    m.Hx  = pyo.Var()
    m.pen1 = pyo.Var(bounds=(0, 20.0))
    m.pen2 = pyo.Var(bounds=(0, 20.0))

    m.dT1 = pyo.Constraint(expr=
        m.T1x == T1 + p['zeta_exch']*(T2-T1) + p['zeta_loss']*(T_out-T1)
               + p['zeta_conv']*m.p1 - p['zeta_cool']*m.v + p['zeta_occ']*occ1)
    m.dT2 = pyo.Constraint(expr=
        m.T2x == T2 + p['zeta_exch']*(T1-T2) + p['zeta_loss']*(T_out-T2)
               + p['zeta_conv']*m.p2 - p['zeta_cool']*m.v + p['zeta_occ']*occ2)
    m.dH  = pyo.Constraint(expr=
        m.Hx  == H + p['eta_occ']*(occ1+occ2) - p['eta_vent']*m.v)

    # pen_r linearises max(0, T_low - T_rx)
    m.p1c = pyo.Constraint(expr=m.pen1 >= T_low - m.T1x)
    m.p2c = pyo.Constraint(expr=m.pen2 >= T_low - m.T2x)

    # Overrule constraints from current state
    if c > 0:                  m.vc  = pyo.Constraint(expr=m.v  == 1)
    if H >= p['H_high']:       m.hc  = pyo.Constraint(expr=m.v  == 1)
    if state.get('y_low_1'):   m.h1l = pyo.Constraint(expr=m.p1 == P_max)
    if state.get('y_low_2'):   m.h2l = pyo.Constraint(expr=m.p2 == P_max)
    if state.get('y_high_1'):  m.h1h = pyo.Constraint(expr=m.p1 == 0.0)
    if state.get('y_high_2'):  m.h2h = pyo.Constraint(expr=m.p2 == 0.0)

    # c_next is linear in v given current c
    # c=0 -> 2v  |  c=1, v forced -> 0  |  c=2, v forced -> 1
    if   c == 0: c_next = 2.0 * m.v
    elif c == 1: c_next = 0.0
    else:        c_next = 1.0

    vfa = (eta[0]*(m.T1x - T_ok)**2 +
           eta[1]*(m.T2x - T_ok)**2 +
           eta[2]*m.Hx +
           eta[3]*c_next +
           eta[4]*m.pen1 +
           eta[5]*m.pen2 +
           eta[6]*1.0)

    m.obj = pyo.Objective(
        expr=price*(m.p1 + m.p2 + p['P_vent']*m.v) + vfa,
        sense=pyo.minimize)

    solver = pyo.SolverFactory('gurobi')
    solver.options['OutputFlag'] = 0
    solver.options['NonConvex']  = 2   # needed for quadratic VFA terms
    result = solver.solve(m)

    if result.solver.termination_condition != pyo.TerminationCondition.optimal:
        return None
    return float(pyo.value(m.obj))


print("Solver function defined.")

In [ ]:
# eta_all[t]  = fitted weights for time t   (index T = terminal zeros)
eta_all = np.zeros((T + 1, N_FEAT))

print("Starting backward pass ...")
print(f"{'t':>3}  {'valid':>6}  {'RMSE':>8}  weights (7 values)")
print("-" * 72)

t0 = time.perf_counter()

for t in range(T - 1, -1, -1):          # t = 9, 8, ..., 0
    eta_next = eta_all[t + 1]            # already fitted (zeros at T)
    targets, rows, n_skip = [], [], 0

    for n in range(N):
        v_star = solve_1step(states[t][n], eta_next)
        if v_star is None:
            n_skip += 1
            continue
        targets.append(v_star)
        rows.append(phi(states[t][n]))

    n_valid = len(targets)
    if n_valid < N_FEAT:
        print(f"  t={t}  only {n_valid} valid samples — carrying forward eta")
        eta_all[t] = eta_all[t + 1]
        continue

    Phi = np.array(rows)       # (n_valid, 7)
    y   = np.array(targets)    # (n_valid,)

    eta_t, _, _, _ = np.linalg.lstsq(Phi, y, rcond=None)
    eta_all[t] = eta_t

    rmse = float(np.sqrt(np.mean((Phi @ eta_t - y)**2)))
    print(f"  {t:>1}  {n_valid:>6}  {rmse:>8.3f}  {np.round(eta_t, 4)}")

print(f"\nBackward pass done in {(time.perf_counter()-t0)/60:.1f} min.")

In [ ]:
os.makedirs('output', exist_ok=True)
weights_path = 'output/adp_weights.json'

weights_dict = {
    'description': 'Time-dependent VFA weights eta_t for t=0..9. '
                   'Produced by ADP_policy_14.ipynb (Forward-Backward algorithm).',
    'feature_names': [
        '(T1-T_ok)^2',
        '(T2-T_ok)^2',
        'H',
        'c_inertia',
        'max(0, T_low-T1)',
        'max(0, T_low-T2)',
        'intercept',
    ],
    'T_ok' : params['T_ok'],
    'T_low': params['T_low'],
    'eta'  : {str(t): eta_all[t].tolist() for t in range(T)},
}

with open(weights_path, 'w') as f:
    json.dump(weights_dict, f, indent=2)

print(f"Weights saved -> '{weights_path}'")
print()
hdr = f"{'t':>2}  {'(T1-ok)^2':>10}  {'(T2-ok)^2':>10}  {'H':>7}  {'c':>5}  {'pen1':>7}  {'pen2':>7}  {'bias':>8}"
print(hdr)
print("-" * len(hdr))
for t in range(T):
    e = eta_all[t]
    print(f"  {t}  {e[0]:>10.4f}  {e[1]:>10.4f}  {e[2]:>7.4f}  {e[3]:>5.3f}  {e[4]:>7.4f}  {e[5]:>7.4f}  {e[6]:>8.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ts  = np.arange(T)
lbs = ['$(T_1{-}T^{ok})^2$', '$(T_2{-}T^{ok})^2$', '$H$', '$c$', 'pen$_1$', 'pen$_2$']
clr = plt.cm.tab10(np.linspace(0, 0.8, 6))

for f in range(6):
    ax1.plot(ts, eta_all[:T, f], marker='o', markersize=4, label=lbs[f], color=clr[f])
ax1.set_title('VFA weights $\\eta_t$ (excl. intercept)')
ax1.set_xlabel('Timestep $t$'); ax1.set_ylabel('Weight')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3); ax1.set_xticks(ts)

ax2.plot(ts, eta_all[:T, 6], marker='s', color='purple')
ax2.set_title('Intercept $\\eta_{t,6}$')
ax2.set_xlabel('Timestep $t$'); ax2.set_ylabel('Intercept')
ax2.grid(True, alpha=0.3); ax2.set_xticks(ts)

plt.tight_layout()
plt.savefig('output/adp_weights_convergence.pdf', bbox_inches='tight')
plt.show()
print("Saved: output/adp_weights_convergence.pdf")

## Verification

`ADP_policy_14.py` loads `output/adp_weights.json` at import time.
Running the cell below confirms the file is readable and the shapes are correct.

In [ ]:
with open('output/adp_weights.json') as f:
    check = json.load(f)

print("JSON keys:", list(check.keys()))
print(f"Timesteps in 'eta': {sorted(check['eta'].keys())}")
print(f"Feature vector length: {len(check['eta']['0'])}")
print()
print("ADP_policy_14.py will load this file and call select_action(state)")
print("which runs solve_adp_step(state, ETA[t], params) for t = state['current_time'].")